In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 104
==================================================
Week: 15 of 24
Day: 104 of 168
Date: May 21, 2026 
Topic: TurtleSim Controller

Week 15 Progress:
✅ Day 99:  ROS2 Setup & TurtleSim
✅ Day 100: ROS2 PubSub & Python Nodes
✅ Day 101: ROS2 Services & Actions
✅ Day 102: ROS2 Packages, Launch Files & Parameters
✅ Day 103: Sensor Data Processing
🔄 Day 104: Autonomous TurtleSim Controller (TODAY!)
⬜ Day 105: TurtleSim Controller — Polish & Demo

Progress: 71.4% (5/7 days)

==================================================
🎯 Week 15 Project: ROS2 & Robotics
- Install and configure ROS2 Humble
- Master nodes, topics, services, and actions
- Build a complete TurtleSim Controller package
- Process sensor data (camera, LiDAR)
- Create demo video and professional README

🎯 Today's Learning Objectives:
1. Build the complete TurtleController class (pub + sub + services)
2. Implement keyboard control (WASD)
3. Implement autonomous boundary detection and avoidance
4. Implement multiple movement patterns
5. Wire everything into a single launch file

📚 Today's Structure:
   Part 1 (2h): Core TurtleController Node
   Part 2 (2h): Keyboard Control & Patterns
   Part 3 (2h): Full Integration & Launch File

🎯 SUCCESS CRITERIA:
   ✅ TurtleController node running with all features
   ✅ WASD keyboard control working
   ✅ Autonomous boundary avoidance working
   ✅ At least 3 movement patterns (circle, square, figure8)
   ✅ Single launch file starts the entire system
   ✅ All nodes communicating correctly

==================================================
"""

In [1]:
print("=" * 80)
print("🚀 TURTLESIM CONTROLLER — Day 104")
print("=" * 80)
print("""
BUILD DAY — everything from Days 99-103 comes together.
Today we build the complete TurtleSim Controller package
from scratch: keyboard control, autonomous movement,
boundary detection, services, and a full launch file.
""")
print("=" * 80)

🚀 TURTLESIM CONTROLLER — Day 104

BUILD DAY — everything from Days 99-103 comes together.
Today we build the complete TurtleSim Controller package
from scratch: keyboard control, autonomous movement,
boundary detection, services, and a full launch file.



In [2]:
print("\n" + "=" * 80)
print("🧠 PART 1: CORE TURTLECONTROLLER NODE")
print("=" * 80)


🧠 PART 1: CORE TURTLECONTROLLER NODE


In [3]:
# ==================================================
# EXERCISE 1.1: ARCHITECTURE DESIGN
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: TurtleSim Controller Architecture")
print("=" * 80)

print("""
📊 TURTLESIM CONTROLLER ARCHITECTURE:

  ┌─────────────────────────────────────────────────────────┐
  │              turtle_controller Package                   │
  │                                                         │
  │  ┌─────────────────────────────────────────────────┐   │
  │  │            TurtleController Node                 │   │
  │  │                                                  │   │
  │  │  Publishers:                                     │   │
  │  │  └── /turtle1/cmd_vel  (Twist)                  │   │
  │  │                                                  │   │
  │  │  Subscribers:                                    │   │
  │  │  └── /turtle1/pose     (Pose)                   │   │
  │  │                                                  │   │
  │  │  Services:                                       │   │
  │  │  ├── /turtle_controller/set_pattern             │   │
  │  │  ├── /turtle_controller/go_home                 │   │
  │  │  └── /turtle_controller/set_mode                │   │
  │  │                                                  │   │
  │  │  Service Clients:                                │   │
  │  │  ├── /turtle1/teleport_absolute                 │   │
  │  │  └── /turtle1/set_pen                           │   │
  │  │                                                  │   │
  │  │  Modes:                                          │   │
  │  │  ├── KEYBOARD  → WASD control                   │   │
  │  │  ├── AUTONOMOUS → boundary-aware movement       │   │
  │  │  └── PATTERN   → circle/square/figure8          │   │
  │  └─────────────────────────────────────────────────┘   │
  │                                                         │
  │  ┌──────────────┐    ┌──────────────────────────────┐  │
  │  │  keyboard_   │    │   turtlesim_node             │  │
  │  │  input.py    │    │   (external package)         │  │
  │  └──────────────┘    └──────────────────────────────┘  │
  └─────────────────────────────────────────────────────────┘

  Data flow:
  keyboard_input → /turtle_controller/key_press → TurtleController
  TurtleController → /turtle1/cmd_vel → turtlesim
  turtlesim → /turtle1/pose → TurtleController (feedback loop)
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: TurtleSim Controller Architecture

📊 TURTLESIM CONTROLLER ARCHITECTURE:

  ┌─────────────────────────────────────────────────────────┐
  │              turtle_controller Package                   │
  │                                                         │
  │  ┌─────────────────────────────────────────────────┐   │
  │  │            TurtleController Node                 │   │
  │  │                                                  │   │
  │  │  Publishers:                                     │   │
  │  │  └── /turtle1/cmd_vel  (Twist)                  │   │
  │  │                                                  │   │
  │  │  Subscribers:                                    │   │
  │  │  └── /turtle1/pose     (Pose)                   │   │
  │  │                                                  │   │
  │  │  Services:                                       │   │
  │  │  ├── /turtle_controller/set_pattern             │   │
  │  │  ├── /turtle_controller/go_home         

In [4]:
# ==================================================
# EXERCISE 1.2: MAIN TURTLE CONTROLLER NODE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Main TurtleController Node")
print("=" * 80)

turtle_controller_py = '''
# ============================================================
# turtle_controller/turtle_controller_node.py
# Main controller node — the heart of the project
#
# Save to: ~/ros2_ws/src/turtle_controller/turtle_controller/
# ============================================================

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist
from turtlesim.msg import Pose
from turtlesim.srv import TeleportAbsolute, SetPen
from std_srvs.srv import Trigger
from std_msgs.msg import String
import math
from enum import Enum


class Mode(Enum):
    """Controller operating modes."""
    AUTONOMOUS = "autonomous"
    KEYBOARD   = "keyboard"
    PATTERN    = "pattern"
    STOPPED    = "stopped"


class Pattern(Enum):
    """Available movement patterns."""
    CIRCLE   = "circle"
    SQUARE   = "square"
    FIGURE8  = "figure8"
    SPIRAL   = "spiral"


class TurtleController(Node):
    """
    Main TurtleSim Controller node.
    Combines autonomous movement, keyboard control,
    boundary detection, and pattern drawing.
    """

    # Canvas boundaries (TurtleSim is 11.08 x 11.08)
    CANVAS_MIN = 0.5
    CANVAS_MAX = 10.58

    def __init__(self):
        super().__init__("turtle_controller")

        # ── Parameters ────────────────────────────────────────
        self.declare_parameter("boundary_margin", 2.0)
        self.declare_parameter("forward_speed", 2.0)
        self.declare_parameter("turn_speed", 1.5)
        self.declare_parameter("initial_mode", "autonomous")
        self.declare_parameter("initial_pattern", "circle")
        self.declare_parameter("home_x", 5.544)
        self.declare_parameter("home_y", 5.544)

        self.boundary_margin = self.get_parameter("boundary_margin").value
        self.forward_speed   = self.get_parameter("forward_speed").value
        self.turn_speed      = self.get_parameter("turn_speed").value
        self.home_x          = self.get_parameter("home_x").value
        self.home_y          = self.get_parameter("home_y").value

        initial_mode    = self.get_parameter("initial_mode").value
        initial_pattern = self.get_parameter("initial_pattern").value

        self.mode    = Mode(initial_mode)
        self.pattern = Pattern(initial_pattern)

        # ── State ─────────────────────────────────────────────
        self.current_pose   = None
        self.near_boundary  = False
        self.step           = 0
        self.keyboard_twist = Twist()   # Latest keyboard command

        # ── Publisher ─────────────────────────────────────────
        self.cmd_pub = self.create_publisher(
            Twist, "/turtle1/cmd_vel", 10
        )

        # ── Subscriber ────────────────────────────────────────
        self.pose_sub = self.create_subscription(
            Pose, "/turtle1/pose", self.pose_callback, 10
        )

        # Subscribe to keyboard input topic
        self.key_sub = self.create_subscription(
            String, "/turtle_controller/key_press",
            self.key_callback, 10
        )

        # ── Service Servers ───────────────────────────────────
        self.go_home_srv = self.create_service(
            Trigger, "/turtle_controller/go_home",
            self.go_home_callback
        )
        self.set_auto_srv = self.create_service(
            Trigger, "/turtle_controller/set_autonomous",
            self.set_autonomous_callback
        )
        self.set_stop_srv = self.create_service(
            Trigger, "/turtle_controller/stop",
            self.stop_callback
        )

        # ── Service Clients ───────────────────────────────────
        self.teleport_client = self.create_client(
            TeleportAbsolute, "/turtle1/teleport_absolute"
        )
        self.set_pen_client = self.create_client(
            SetPen, "/turtle1/set_pen"
        )

        # ── Control Loop Timer (10 Hz) ─────────────────────────
        self.control_timer = self.create_timer(0.1, self.control_loop)

        # ── Status Timer (1 Hz) ───────────────────────────────
        self.status_timer = self.create_timer(1.0, self.log_status)

        self.get_logger().info(f"""
╔══════════════════════════════════════╗
║    TurtleSim Controller Started!    ║
╠══════════════════════════════════════╣
║  Mode:    {self.mode.value:<28}║
║  Pattern: {self.pattern.value:<28}║
║  Speed:   {self.forward_speed:<28}║
║  Margin:  {self.boundary_margin:<28}║
╚══════════════════════════════════════╝
""")

    # ── Subscriber Callbacks ──────────────────────────────────
    def pose_callback(self, msg: Pose):
        """Update state from turtle pose."""
        self.current_pose = msg

        near_x = (
            msg.x < self.CANVAS_MIN + self.boundary_margin or
            msg.x > self.CANVAS_MAX - self.boundary_margin
        )
        near_y = (
            msg.y < self.CANVAS_MIN + self.boundary_margin or
            msg.y > self.CANVAS_MAX - self.boundary_margin
        )
        self.near_boundary = near_x or near_y

    def key_callback(self, msg: String):
        """Handle keyboard input from keyboard_input node."""
        key = msg.data.strip().lower()
        twist = Twist()

        key_map = {
            "w": (self.forward_speed, 0.0),
            "s": (-self.forward_speed, 0.0),
            "a": (0.0, self.turn_speed),
            "d": (0.0, -self.turn_speed),
            " ": (0.0, 0.0),            # Space = stop
        }

        if key in key_map:
            twist.linear.x, twist.angular.z = key_map[key]
            self.keyboard_twist = twist

            # Switch to keyboard mode on any key press
            if self.mode != Mode.KEYBOARD:
                self.mode = Mode.KEYBOARD
                self.get_logger().info("Switched to KEYBOARD mode")

        elif key == "m":
            self.cycle_mode()
        elif key == "p":
            self.cycle_pattern()

    # ── Control Loop ──────────────────────────────────────────
    def control_loop(self):
        """Main control logic — runs at 10 Hz."""
        if self.current_pose is None:
            return

        msg = Twist()

        if self.mode == Mode.STOPPED:
            pass   # msg is all zeros — stop

        elif self.mode == Mode.KEYBOARD:
            msg = self.keyboard_twist

        elif self.mode == Mode.AUTONOMOUS:
            msg = self.autonomous_control()

        elif self.mode == Mode.PATTERN:
            msg = self.pattern_control()
            self.step += 1

        self.cmd_pub.publish(msg)

    def autonomous_control(self) -> Twist:
        """
        Boundary-aware autonomous movement.
        Move forward until near boundary, then turn.
        """
        msg = Twist()
        if self.near_boundary:
            msg.linear.x = 0.0
            msg.angular.z = self.turn_speed
        else:
            msg.linear.x = self.forward_speed
            msg.angular.z = 0.0
        return msg

    def pattern_control(self) -> Twist:
        """Execute the current movement pattern."""
        msg = Twist()

        if self.pattern == Pattern.CIRCLE:
            msg.linear.x  = self.forward_speed
            msg.angular.z = 1.0

        elif self.pattern == Pattern.SQUARE:
            phase = self.step % 30
            if phase < 20:
                msg.linear.x  = self.forward_speed
                msg.angular.z = 0.0
            else:
                msg.linear.x  = 0.0
                msg.angular.z = math.pi / 2

        elif self.pattern == Pattern.FIGURE8:
            phase = self.step % 200
            msg.linear.x  = self.forward_speed
            msg.angular.z = 1.0 if phase < 100 else -1.0

        elif self.pattern == Pattern.SPIRAL:
            # Gradually increasing radius
            t = self.step * 0.1
            msg.linear.x  = min(0.5 + t * 0.05, self.forward_speed)
            msg.angular.z = max(2.0 - t * 0.02, 0.3)

        return msg

    # ── Service Callbacks ─────────────────────────────────────
    def go_home_callback(self, request, response):
        """Teleport turtle to home position."""
        if self.teleport_client.wait_for_service(timeout_sec=2.0):
            req = TeleportAbsolute.Request()
            req.x = self.home_x
            req.y = self.home_y
            req.theta = 0.0
            future = self.teleport_client.call_async(req)
            rclpy.spin_until_future_complete(self, future, timeout_sec=2.0)
            response.success = True
            response.message = f"Turtle sent home to ({self.home_x}, {self.home_y})"
        else:
            response.success = False
            response.message = "Teleport service unavailable"
        return response

    def set_autonomous_callback(self, request, response):
        """Switch to autonomous mode."""
        self.mode = Mode.AUTONOMOUS
        self.step = 0
        response.success = True
        response.message = "Switched to AUTONOMOUS mode"
        self.get_logger().info(response.message)
        return response

    def stop_callback(self, request, response):
        """Stop all movement."""
        self.mode = Mode.STOPPED
        stop_msg = Twist()
        self.cmd_pub.publish(stop_msg)
        response.success = True
        response.message = "Turtle stopped"
        self.get_logger().info(response.message)
        return response

    # ── Helper Methods ────────────────────────────────────────
    def cycle_mode(self):
        """Cycle through available modes."""
        modes = [Mode.AUTONOMOUS, Mode.PATTERN, Mode.KEYBOARD, Mode.STOPPED]
        current_idx = modes.index(self.mode)
        self.mode = modes[(current_idx + 1) % len(modes)]
        self.step = 0
        self.get_logger().info(f"Mode → {self.mode.value}")

    def cycle_pattern(self):
        """Cycle through available patterns."""
        patterns = list(Pattern)
        current_idx = patterns.index(self.pattern)
        self.pattern = patterns[(current_idx + 1) % len(patterns)]
        self.step = 0
        self.get_logger().info(f"Pattern → {self.pattern.value}")

    def log_status(self):
        """Log current status every second."""
        if self.current_pose:
            self.get_logger().info(
                f"Mode: {self.mode.value} | "
                f"Pattern: {self.pattern.value} | "
                f"Pos: ({self.current_pose.x:.1f}, {self.current_pose.y:.1f}) | "
                f"Boundary: {self.near_boundary}"
            )


def main(args=None):
    rclpy.init(args=args)
    node = TurtleController()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        stop_msg = Twist()
        node.cmd_pub.publish(stop_msg)
        node.get_logger().info("TurtleController stopped")
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

print("📄 turtle_controller/turtle_controller_node.py:")
print(turtle_controller_py)

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Main TurtleController Node
📄 turtle_controller/turtle_controller_node.py:

# ============================================================
# turtle_controller/turtle_controller_node.py
# Main controller node — the heart of the project
#
# Save to: ~/ros2_ws/src/turtle_controller/turtle_controller/
# ============================================================

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist
from turtlesim.msg import Pose
from turtlesim.srv import TeleportAbsolute, SetPen
from std_srvs.srv import Trigger
from std_msgs.msg import String
import math
from enum import Enum


class Mode(Enum):
    """Controller operating modes."""
    AUTONOMOUS = "autonomous"
    KEYBOARD   = "keyboard"
    PATTERN    = "pattern"
    STOPPED    = "stopped"


class Pattern(Enum):
    """Available movement patterns."""
    CIRCLE   = "circle"
    SQUARE   = "square"
    FIGURE8  = "figure8"
    SPIRAL   = "spiral"


class TurtleController(Node):
    

In [5]:
print("\n" + "=" * 80)
print("⌨️  PART 2: KEYBOARD CONTROL & PEN COLORS")
print("=" * 80)


⌨️  PART 2: KEYBOARD CONTROL & PEN COLORS


In [6]:
# ==================================================
# EXERCISE 2.1: KEYBOARD INPUT NODE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: Keyboard Input Node")
print("=" * 80)

"""
📖 THEORY: Keyboard Input in ROS2

Reading keyboard input in ROS2 requires a dedicated node
that reads from stdin and publishes key presses as messages.
This decouples input handling from the controller logic.
"""

keyboard_input_py = '''
# ============================================================
# turtle_controller/keyboard_input.py
# Reads keyboard input and publishes to /turtle_controller/key_press
#
# Save to: ~/ros2_ws/src/turtle_controller/turtle_controller/
# ============================================================

import rclpy
from rclpy.node import Node
from std_msgs.msg import String
import sys
import tty
import termios
import threading


class KeyboardInput(Node):
    """
    Reads keyboard input non-blocking and publishes key presses.
    Run this in a terminal with focus to control the turtle.
    """

    def __init__(self):
        super().__init__("keyboard_input")

        self.publisher = self.create_publisher(
            String,
            "/turtle_controller/key_press",
            10,
        )

        self.get_logger().info("""
╔══════════════════════════════════════╗
║       Keyboard Control Active       ║
╠══════════════════════════════════════╣
║  W / ↑  → Move Forward             ║
║  S / ↓  → Move Backward            ║
║  A / ←  → Turn Left                ║
║  D / →  → Turn Right               ║
║  SPACE  → Stop                     ║
║  M      → Cycle Mode               ║
║  P      → Cycle Pattern            ║
║  Q      → Quit                     ║
╚══════════════════════════════════════╝
""")

        # Start keyboard reading in a background thread
        self.running = True
        self.key_thread = threading.Thread(
            target=self.read_keys, daemon=True
        )
        self.key_thread.start()

    def get_key(self):
        """Read a single keypress without waiting for Enter."""
        fd = sys.stdin.fileno()
        old_settings = termios.tcgetattr(fd)
        try:
            tty.setraw(fd)
            key = sys.stdin.read(1)
        finally:
            termios.tcsetattr(fd, termios.TCSADRAIN, old_settings)
        return key

    def read_keys(self):
        """Background thread: continuously read and publish keys."""
        while self.running:
            try:
                key = self.get_key()

                if key == "q":
                    self.get_logger().info("Quit signal received")
                    self.running = False
                    rclpy.shutdown()
                    break

                # Handle arrow keys (escape sequences)
                if key == "\\x1b":
                    key2 = sys.stdin.read(1)
                    key3 = sys.stdin.read(1)
                    arrow_map = {
                        "A": "w",   # Up arrow → W
                        "B": "s",   # Down arrow → S
                        "C": "d",   # Right arrow → D
                        "D": "a",   # Left arrow → A
                    }
                    key = arrow_map.get(key3, "")

                if key:
                    msg = String()
                    msg.data = key
                    self.publisher.publish(msg)

            except Exception as e:
                self.get_logger().error(f"Key read error: {e}")
                break


def main(args=None):
    rclpy.init(args=args)
    node = KeyboardInput()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.running = False
        node.destroy_node()
        rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

print("📄 turtle_controller/keyboard_input.py:")
print(keyboard_input_py)

print("""
📋 HOW KEYBOARD CONTROL WORKS:
─────────────────────────────────────────────────────────────
  keyboard_input node:
  ├── Reads raw keypresses from terminal (non-blocking)
  ├── Publishes key as String to /turtle_controller/key_press
  └── Runs in background thread (doesn't block rclpy.spin)

  turtle_controller node:
  ├── Subscribes to /turtle_controller/key_press
  ├── Maps keys to Twist commands
  └── Switches to KEYBOARD mode on first keypress

  This decoupled design means:
  ✅ Controller logic stays clean
  ✅ Keyboard node can be replaced with joystick/gamepad node
  ✅ Same controller works with any input source
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: Keyboard Input Node
📄 turtle_controller/keyboard_input.py:

# ============================================================
# turtle_controller/keyboard_input.py
# Reads keyboard input and publishes to /turtle_controller/key_press
#
# Save to: ~/ros2_ws/src/turtle_controller/turtle_controller/
# ============================================================

import rclpy
from rclpy.node import Node
from std_msgs.msg import String
import sys
import tty
import termios
import threading


class KeyboardInput(Node):
    """
    Reads keyboard input non-blocking and publishes key presses.
    Run this in a terminal with focus to control the turtle.
    """

    def __init__(self):
        super().__init__("keyboard_input")

        self.publisher = self.create_publisher(
            String,
            "/turtle_controller/key_press",
            10,
        )

        self.get_logger().info("""
╔══════════════════════════════════════╗
║       Keyboard Control Active       ║
╠════

In [7]:
# ==================================================
# EXERCISE 2.2: PEN COLOR MANAGER
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Pen Color Manager")
print("=" * 80)

pen_colors_py = '''
# ============================================================
# turtle_controller/pen_manager.py
# Changes pen color based on current movement pattern
#
# Save to: ~/ros2_ws/src/turtle_controller/turtle_controller/
# ============================================================

import rclpy
from rclpy.node import Node
from std_msgs.msg import String
from turtlesim.srv import SetPen


# Pattern → pen color mapping
PATTERN_COLORS = {
    "circle":  {"r": 255, "g": 100, "b": 100, "width": 2},  # Red
    "square":  {"r": 100, "g": 255, "b": 100, "width": 3},  # Green
    "figure8": {"r": 100, "g": 100, "b": 255, "width": 2},  # Blue
    "spiral":  {"r": 255, "g": 200, "b": 50,  "width": 1},  # Yellow
    "autonomous": {"r": 200, "g": 200, "b": 200, "width": 2}, # Gray
    "keyboard": {"r": 255, "g": 255, "b": 255, "width": 2},  # White
}


class PenManager(Node):
    """
    Listens for mode/pattern changes and updates pen color accordingly.
    """

    def __init__(self):
        super().__init__("pen_manager")

        # Subscribe to status topic from controller
        self.status_sub = self.create_subscription(
            String,
            "/turtle_controller/status",
            self.status_callback,
            10,
        )

        # Client to call TurtleSim set_pen service
        self.set_pen_client = self.create_client(
            SetPen, "/turtle1/set_pen"
        )

        self.current_pattern = None
        self.get_logger().info("PenManager started — automatic pen colors!")

    def status_callback(self, msg: String):
        """Update pen color when pattern changes."""
        # Status format: "mode:autonomous pattern:circle"
        parts = dict(p.split(":") for p in msg.data.split())
        pattern = parts.get("pattern", "autonomous")
        mode = parts.get("mode", "autonomous")

        # Use mode as key if in keyboard/autonomous, pattern otherwise
        color_key = pattern if mode == "pattern" else mode

        if color_key != self.current_pattern:
            self.current_pattern = color_key
            self.set_pen_color(color_key)

    def set_pen_color(self, key: str):
        """Call set_pen service with color for the given key."""
        if not self.set_pen_client.wait_for_service(timeout_sec=1.0):
            return

        color = PATTERN_COLORS.get(key, PATTERN_COLORS["autonomous"])

        req = SetPen.Request()
        req.r     = color["r"]
        req.g     = color["g"]
        req.b     = color["b"]
        req.width = color["width"]
        req.off   = 0

        future = self.set_pen_client.call_async(req)
        self.get_logger().info(
            f"Pen color → {key} "
            f"(R:{color[\'r\']} G:{color[\'g\']} B:{color[\'b\']})"
        )


def main(args=None):
    rclpy.init(args=args)
    node = PenManager()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

print("📄 turtle_controller/pen_manager.py:")
print(pen_colors_py)

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Pen Color Manager
📄 turtle_controller/pen_manager.py:

# ============================================================
# turtle_controller/pen_manager.py
# Changes pen color based on current movement pattern
#
# Save to: ~/ros2_ws/src/turtle_controller/turtle_controller/
# ============================================================

import rclpy
from rclpy.node import Node
from std_msgs.msg import String
from turtlesim.srv import SetPen


# Pattern → pen color mapping
PATTERN_COLORS = {
    "circle":  {"r": 255, "g": 100, "b": 100, "width": 2},  # Red
    "square":  {"r": 100, "g": 255, "b": 100, "width": 3},  # Green
    "figure8": {"r": 100, "g": 100, "b": 255, "width": 2},  # Blue
    "spiral":  {"r": 255, "g": 200, "b": 50,  "width": 1},  # Yellow
    "autonomous": {"r": 200, "g": 200, "b": 200, "width": 2}, # Gray
    "keyboard": {"r": 255, "g": 255, "b": 255, "width": 2},  # White
}


class PenManager(Node):
    """
    Listens for mode/pattern changes and updates 

In [8]:
print("\n" + "=" * 80)
print("🔧 PART 3: FULL INTEGRATION & LAUNCH FILE")
print("=" * 80)


🔧 PART 3: FULL INTEGRATION & LAUNCH FILE


In [9]:
# ==================================================
# EXERCISE 3.1: COMPLETE LAUNCH FILE
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Complete Launch File")
print("=" * 80)

complete_launch_py = '''
# ============================================================
# launch/turtle_controller.launch.py — Complete Launch File
# Starts all nodes: TurtleSim + Controller + Keyboard + Pen
#
# Run with:
#   ros2 launch turtle_controller turtle_controller.launch.py
#
# With custom args:
#   ros2 launch turtle_controller turtle_controller.launch.py \\
#     mode:=pattern pattern:=square boundary_margin:=3.0
# ============================================================

from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument, LogInfo, TimerAction
from launch.substitutions import LaunchConfiguration
from launch_ros.actions import Node
import os
from ament_index_python.packages import get_package_share_directory


def generate_launch_description():

    pkg_dir = get_package_share_directory("turtle_controller")
    config  = os.path.join(pkg_dir, "config", "params.yaml")

    # ── Launch Arguments ──────────────────────────────────────
    args = [
        DeclareLaunchArgument(
            "mode", default_value="autonomous",
            description="Initial mode: autonomous, pattern, keyboard, stopped"
        ),
        DeclareLaunchArgument(
            "pattern", default_value="circle",
            description="Initial pattern: circle, square, figure8, spiral"
        ),
        DeclareLaunchArgument(
            "boundary_margin", default_value="2.0",
            description="Distance from wall to start turning"
        ),
        DeclareLaunchArgument(
            "forward_speed", default_value="2.0",
            description="Forward movement speed"
        ),
        DeclareLaunchArgument(
            "turn_speed", default_value="1.5",
            description="Turning speed"
        ),
    ]

    # ── Nodes ─────────────────────────────────────────────────
    turtlesim_node = Node(
        package="turtlesim",
        executable="turtlesim_node",
        name="turtlesim",
        output="screen",
    )

    controller_node = Node(
        package="turtle_controller",
        executable="turtle_controller_node",
        name="turtle_controller",
        output="screen",
        parameters=[
            config,
            {
                "initial_mode":    LaunchConfiguration("mode"),
                "initial_pattern": LaunchConfiguration("pattern"),
                "boundary_margin": LaunchConfiguration("boundary_margin"),
                "forward_speed":   LaunchConfiguration("forward_speed"),
                "turn_speed":      LaunchConfiguration("turn_speed"),
            },
        ],
    )

    keyboard_node = Node(
        package="turtle_controller",
        executable="keyboard_input",
        name="keyboard_input",
        output="screen",
        prefix="xterm -e",   # Open in separate terminal window
    )

    pen_manager_node = Node(
        package="turtle_controller",
        executable="pen_manager",
        name="pen_manager",
        output="screen",
    )

    # ── Delayed start (wait for turtlesim to be ready) ────────
    delayed_controller = TimerAction(
        period=2.0,          # Wait 2 seconds before starting controller
        actions=[controller_node, pen_manager_node],
    )

    delayed_keyboard = TimerAction(
        period=3.0,          # Wait 3 seconds before keyboard node
        actions=[keyboard_node],
    )

    return LaunchDescription(
        args + [
            LogInfo(msg="Launching TurtleSim Controller System..."),
            turtlesim_node,
            delayed_controller,
            delayed_keyboard,
        ]
    )
'''

print("📄 launch/turtle_controller.launch.py:")
print(complete_launch_py)

print("""
📋 HOW TO BUILD AND RUN:
─────────────────────────────────────────────────────────────
  # 1. Update setup.py entry_points to include all new nodes:
  #    "turtle_controller_node = turtle_controller.turtle_controller_node:main",
  #    "keyboard_input = turtle_controller.keyboard_input:main",
  #    "pen_manager = turtle_controller.pen_manager:main",

  # 2. Build the package
  cd ~/ros2_ws
  colcon build --packages-select turtle_controller --symlink-install
  source install/setup.bash

  # 3. Launch everything!
  ros2 launch turtle_controller turtle_controller.launch.py

  # 4. With custom settings:
  ros2 launch turtle_controller turtle_controller.launch.py \\
    mode:=pattern pattern:=square boundary_margin:=3.0

  # 5. Test individual services:
  ros2 service call /turtle_controller/go_home std_srvs/srv/Trigger
  ros2 service call /turtle_controller/stop std_srvs/srv/Trigger
  ros2 service call /turtle_controller/set_autonomous std_srvs/srv/Trigger
""")

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Complete Launch File
📄 launch/turtle_controller.launch.py:

# ============================================================
# launch/turtle_controller.launch.py — Complete Launch File
# Starts all nodes: TurtleSim + Controller + Keyboard + Pen
#
# Run with:
#   ros2 launch turtle_controller turtle_controller.launch.py
#
# With custom args:
#   ros2 launch turtle_controller turtle_controller.launch.py \
#     mode:=pattern pattern:=square boundary_margin:=3.0
# ============================================================

from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument, LogInfo, TimerAction
from launch.substitutions import LaunchConfiguration
from launch_ros.actions import Node
import os
from ament_index_python.packages import get_package_share_directory


def generate_launch_description():

    pkg_dir = get_package_share_directory("turtle_controller")
    config  = os.path.join(pkg_dir, "config", "params.yaml")

    # ── Launch Argum

In [10]:
# ==================================================
# EXERCISE 3.2: UPDATED SETUP.PY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: Updated setup.py with All Nodes")
print("=" * 80)

updated_setup_py = '''
# ============================================================
# setup.py — Updated with all Day 104 nodes
# ============================================================

from setuptools import setup, find_packages
import os
from glob import glob

package_name = "turtle_controller"

setup(
    name=package_name,
    version="0.2.0",
    packages=find_packages(exclude=["test"]),
    data_files=[
        ("share/ament_index/resource_index/packages",
         ["resource/" + package_name]),
        ("share/" + package_name, ["package.xml"]),
        (os.path.join("share", package_name, "launch"),
         glob("launch/*.launch.py")),
        (os.path.join("share", package_name, "config"),
         glob("config/*.yaml")),
    ],
    install_requires=["setuptools"],
    zip_safe=True,
    maintainer="Audrey",
    maintainer_email="audrey@example.com",
    description="Autonomous TurtleSim Controller — Week 15 ML Journey",
    license="MIT",
    entry_points={
        "console_scripts": [
            # Day 100
            "turtle_mover = turtle_controller.turtle_mover:main",
            "pose_listener = turtle_controller.pose_listener:main",
            "smart_mover = turtle_controller.smart_mover:main",
            # Day 101
            "turtle_commander_server = turtle_controller.turtle_commander_server:main",
            "rotate_client = turtle_controller.rotate_client:main",
            # Day 103
            "camera_processor = turtle_controller.camera_processor:main",
            "lidar_processor = turtle_controller.lidar_processor:main",
            # Day 104
            "turtle_controller_node = turtle_controller.turtle_controller_node:main",
            "keyboard_input = turtle_controller.keyboard_input:main",
            "pen_manager = turtle_controller.pen_manager:main",
        ],
    },
)
'''

print("📄 setup.py (updated):")
print(updated_setup_py)

print("""
📂 COMPLETE PACKAGE STRUCTURE AFTER DAY 104:
─────────────────────────────────────────────────────────────
  ~/ros2_ws/src/turtle_controller/
  ├── turtle_controller/
  │   ├── __init__.py
  │   ├── turtle_mover.py              ← Day 100
  │   ├── pose_listener.py             ← Day 100
  │   ├── smart_mover.py               ← Day 100/102
  │   ├── turtle_commander_server.py   ← Day 101
  │   ├── rotate_client.py             ← Day 101
  │   ├── camera_processor.py          ← Day 103
  │   ├── lidar_processor.py           ← Day 103
  │   ├── turtle_controller_node.py    ← Day 104 ★ Main
  │   ├── keyboard_input.py            ← Day 104 ★ Input
  │   └── pen_manager.py               ← Day 104 ★ Visual
  │
  ├── launch/
  │   ├── turtlesim_basic.launch.py    ← Day 102
  │   ├── turtlesim_full.launch.py     ← Day 102
  │   └── turtle_controller.launch.py  ← Day 104 ★ Complete
  │
  ├── config/
  │   └── params.yaml                  ← Day 102
  │
  ├── package.xml
  └── setup.py                         ← Updated today
""")

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: Updated setup.py with All Nodes
📄 setup.py (updated):

# ============================================================
# setup.py — Updated with all Day 104 nodes
# ============================================================

from setuptools import setup, find_packages
import os
from glob import glob

package_name = "turtle_controller"

setup(
    name=package_name,
    version="0.2.0",
    packages=find_packages(exclude=["test"]),
    data_files=[
        ("share/ament_index/resource_index/packages",
         ["resource/" + package_name]),
        ("share/" + package_name, ["package.xml"]),
        (os.path.join("share", package_name, "launch"),
         glob("launch/*.launch.py")),
        (os.path.join("share", package_name, "config"),
         glob("config/*.yaml")),
    ],
    install_requires=["setuptools"],
    zip_safe=True,
    maintainer="Audrey",
    maintainer_email="audrey@example.com",
    description="Autonomous TurtleSim Controller — Week 15 ML Journey",


In [11]:
# ==================================================
# EXERCISE 3.3: DAY 104 SUMMARY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.3: Day 104 Summary")
print("=" * 80)

print("""
📚 WHAT WE BUILT TODAY:

✅ TurtleController Node (turtle_controller_node.py):
   • Mode system: AUTONOMOUS, KEYBOARD, PATTERN, STOPPED
   • Pattern system: CIRCLE, SQUARE, FIGURE8, SPIRAL
   • Pose subscriber → boundary detection → control loop
   • Service servers: go_home, set_autonomous, stop
   • Service clients: teleport_absolute, set_pen
   • Status logging every second
   • Enum-based mode and pattern management

✅ Keyboard Input Node (keyboard_input.py):
   • Non-blocking key reading via tty/termios
   • Background thread for continuous input
   • WASD + arrow keys + Space + M + P + Q
   • Publishes to /turtle_controller/key_press topic
   • Decoupled from controller (swappable with joystick)

✅ Pen Manager Node (pen_manager.py):
   • Subscribes to /turtle_controller/status
   • Automatically changes pen color per mode/pattern
   • Circle=Red, Square=Green, Figure8=Blue, Spiral=Yellow

✅ Complete Launch File (turtle_controller.launch.py):
   • All nodes with TimerAction delays
   • DeclareLaunchArgument for CLI customization
   • Loads from config/params.yaml
   • xterm for keyboard in separate window

💡 KEY DESIGN DECISIONS:
   1. Mode + Pattern as Enums → type-safe, easy to extend
   2. pose_callback updates state, control_loop reads state
      → Clean separation, no race conditions
   3. Keyboard node is separate → controller works without keyboard
   4. TimerAction delays → nodes start in correct order
   5. All parameters declared → fully configurable from CLI/YAML
""")

print("\n✅ Exercise 3.3 Complete!")
print("=" * 80)


EXERCISE 3.3: Day 104 Summary

📚 WHAT WE BUILT TODAY:

✅ TurtleController Node (turtle_controller_node.py):
   • Mode system: AUTONOMOUS, KEYBOARD, PATTERN, STOPPED
   • Pattern system: CIRCLE, SQUARE, FIGURE8, SPIRAL
   • Pose subscriber → boundary detection → control loop
   • Service servers: go_home, set_autonomous, stop
   • Service clients: teleport_absolute, set_pen
   • Status logging every second
   • Enum-based mode and pattern management

✅ Keyboard Input Node (keyboard_input.py):
   • Non-blocking key reading via tty/termios
   • Background thread for continuous input
   • WASD + arrow keys + Space + M + P + Q
   • Publishes to /turtle_controller/key_press topic
   • Decoupled from controller (swappable with joystick)

✅ Pen Manager Node (pen_manager.py):
   • Subscribes to /turtle_controller/status
   • Automatically changes pen color per mode/pattern
   • Circle=Red, Square=Green, Figure8=Blue, Spiral=Yellow

✅ Complete Launch File (turtle_controller.launch.py):
   • All

In [12]:
print("\n" + "=" * 80)
print("🎯 DAY 104 COMPLETE! ✅")
print("=" * 80)

print("""
OBJECTIVES ACHIEVED:
   ✅ Built TurtleController with 4 modes and 4 patterns
   ✅ Keyboard control via dedicated input node (WASD + arrows)
   ✅ Autonomous boundary detection and avoidance
   ✅ Pen color manager for visual pattern differentiation
   ✅ Complete launch file with all nodes, delays, and CLI args
   ✅ Updated setup.py with all 11 console script entry points

💡 KEY FILES CREATED TODAY:
   turtle_controller/
   ├── turtle_controller_node.py  ← Main controller (modes + patterns)
   ├── keyboard_input.py          ← Non-blocking keyboard input
   └── pen_manager.py             ← Automatic pen colors

   launch/
   └── turtle_controller.launch.py ← Complete system launch

🎯 TOMORROW (DAY 105) — POLISH & DEMO:
   - Test and debug the full system
   - Add README.md with setup instructions
   - Record demo video showing all features
   - Final package cleanup and documentation
   - Week 15 complete reflection

Week 15 Progress: 85.7% (6/7 days complete)
Overall Progress: Day 104/168 (61.9% complete)
""")

print("=" * 80)


🎯 DAY 104 COMPLETE! ✅

OBJECTIVES ACHIEVED:
   ✅ Built TurtleController with 4 modes and 4 patterns
   ✅ Keyboard control via dedicated input node (WASD + arrows)
   ✅ Autonomous boundary detection and avoidance
   ✅ Pen color manager for visual pattern differentiation
   ✅ Complete launch file with all nodes, delays, and CLI args
   ✅ Updated setup.py with all 11 console script entry points

💡 KEY FILES CREATED TODAY:
   turtle_controller/
   ├── turtle_controller_node.py  ← Main controller (modes + patterns)
   ├── keyboard_input.py          ← Non-blocking keyboard input
   └── pen_manager.py             ← Automatic pen colors

   launch/
   └── turtle_controller.launch.py ← Complete system launch

🎯 TOMORROW (DAY 105) — POLISH & DEMO:
   - Test and debug the full system
   - Add README.md with setup instructions
   - Record demo video showing all features
   - Final package cleanup and documentation
   - Week 15 complete reflection

Week 15 Progress: 85.7% (6/7 days complete)
Overa